# ADP1 Proteomics Fold Change Flux Analysis

## Objectives

This notebook performs flux analysis based on proteomics fold changes between strains.
Using ADP1 wild-type as the reference condition, we compute fold changes for each strain
and fit model fluxes to match those fold changes.

### Analysis Goals:
1. Load proteomics data (log2 format) and average replicates
2. Load reference flux distribution from lactate condition mutant phenotype analysis
3. For each strain (target condition), compute fold changes relative to ADP1 (reference)
4. Use MSExpression.fit_flux_to_proteomics_fold_change_data to fit fluxes
5. Analyze which protein fold changes could/could not be implemented
6. Compare flux changes across strains

### Strains (Target Conditions):
- **ACN2586**: Initial construct
- **ACN2821**: Evolved strain
- **ACN3425, ACN3427, ACN3429, ACN3430**: Additional strain variants

### Reference Condition:
- **ADP1**: Wild-type Acinetobacter baylyi ADP1

## Load Proteomics Data and Average Replicates

This step:
1. Loads proteomics data from Excel spreadsheet (log2 format)
2. Identifies all strain conditions including ADP1 reference
3. Averages replicates while keeping data in log2 format
4. Saves averaged expression data for subsequent analysis

In [1]:
%run util.py

# Load proteomics data in log2 format
raw_expression = MSExpression.from_spreadsheet(
    filename="data/ASCR_UGA_Proteomics_DgoA_add_strains_2025_pyruvate.xlsx",
    sheet_name="Imputed",
    skiprows=0,
    type="Log2",
    id_column="ACIAD"
)

# Define all strain conditions (including ADP1 reference)
all_strains = [
    "Pyruvate_ACN2586_DgoA_2025",
    "Pyruvate_ACN2821_DgoA_2025",
    "Pyruvate_ACN3425_DgoA_2025",
    "Pyruvate_ACN3427_DgoA_2025",
    "Pyruvate_ACN3429_DgoA_2025",
    "Pyruvate_ACN3430_DgoA_2025",
    "Pyruvate_ADP1_DgoA_2025"
]

# Reference condition (ADP1 wild-type)
reference_condition = "Pyruvate_ADP1_DgoA_2025"

# Target conditions (all strains except ADP1)
target_conditions = [s for s in all_strains if s != reference_condition]

print(f"Reference condition: {reference_condition}")
print(f"Target conditions: {target_conditions}")
print(f"Total features (genes): {len(raw_expression.features)}")

# Average replicates while keeping log2 format
averaged_expression = raw_expression.average_expression_replicates(all_strains)

print(f"\nAfter averaging replicates:")
print(f"  Conditions: {[c.id for c in averaged_expression.conditions]}")
print(f"  Data type: {averaged_expression.type}")

# Save strain info and averaged data
util.save("fold_change_strains", {
    "all_strains": all_strains,
    "reference_condition": reference_condition,
    "target_conditions": target_conditions
})
util.save("fold_change_averaged_expression", averaged_expression._data.to_dict())

print("\nSaved strain info and averaged expression data to datacache/")

/Users/chenry/Dropbox/Projects/KBUtilLib/src


[KBUtilLib] Failed to import rcsb_pdb_utils: ModuleNotFoundError: No module named 'aiohttp'


modelseedpy 0.4.2


2026-03-02 16:11:14,913 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-03-02 16:11:14,914 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-03-02 16:11:14,915 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token


loading biochemistry database from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


2026-03-02 16:11:19,941 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase
2026-03-02 16:11:20,328 - __main__.NotebookUtil - ERROR - Failed to import required modules: No module named 'cobrakbase'


ModuleNotFoundError: No module named 'cobrakbase'

Reference condition: Pyruvate_ADP1_DgoA_2025
Target conditions: ['Pyruvate_ACN2586_DgoA_2025', 'Pyruvate_ACN2821_DgoA_2025', 'Pyruvate_ACN3425_DgoA_2025', 'Pyruvate_ACN3427_DgoA_2025', 'Pyruvate_ACN3429_DgoA_2025', 'Pyruvate_ACN3430_DgoA_2025']
Total features (genes): 2383

After averaging replicates:
  Conditions: ['Pyruvate_ACN2586_DgoA_2025', 'Pyruvate_ACN2821_DgoA_2025', 'Pyruvate_ACN3425_DgoA_2025', 'Pyruvate_ACN3427_DgoA_2025', 'Pyruvate_ACN3429_DgoA_2025', 'Pyruvate_ACN3430_DgoA_2025', 'Pyruvate_ADP1_DgoA_2025']
  Data type: Log2


/opt/anaconda3/envs/modelseed/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


NameError: name 'util' is not defined

## Load Reference Flux Distribution

This step:
1. Loads the mutant phenotype analysis results from lactate condition
2. Extracts the flux distribution to use as the reference baseline
3. This flux will be scaled by proteomics fold changes to compute target fluxes

In [1]:
%run util.py

# Load the mutant phenotype analysis results
mutant_phenotype_data = util.load("ADP1-NR-MGR-MGR-Fit0.01,0.9,0.95",notebook_name="ADP1MutantPhenotypeAnalysis")

# Extract lactate condition flux as reference
lactate_data = mutant_phenotype_data["lactate"]
reference_flux = lactate_data["fluxes"]

print(f"Loaded reference flux from lactate condition:")
print(f"  Total reactions with flux data: {len(reference_flux)}")
print(f"  Non-zero fluxes: {sum(1 for v in reference_flux.values() if abs(v) > 1e-9)}")
print(f"  Growth rate: {lactate_data.get('growth_rate', 'N/A')}")

# Show some example fluxes
print(f"\nSample fluxes (first 10):")
for i, (rxn_id, flux) in enumerate(list(reference_flux.items())[:10]):
    print(f"  {rxn_id}: {flux:.6f}")

# Save reference flux for later use
util.save("fold_change_reference_flux", reference_flux)

print("\nSaved reference flux to datacache/")

/home/chenry/Dropbox/Projects/KBUtilLib/src
modelseedpy 0.4.2


2026-01-30 19:56:41,635 - __main__.NotebookUtil - INFO - Loaded configuration from: /home/chenry/.kbutillib/config.yaml
2026-01-30 19:56:41,636 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /home/chenry/.kbase/token


loading biochemistry database from /home/chenry/Dropbox/Projects/ModelSEEDDatabase


2026-01-30 19:56:49,735 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /home/chenry/Dropbox/Projects/ModelSEEDDatabase
2026-01-30 19:56:50,382 - __main__.NotebookUtil - INFO - BLAST tools are available
2026-01-30 19:56:50,383 - __main__.NotebookUtil - INFO - Notebook name: ADP1FoldChangeAnalysis
2026-01-30 19:56:50,384 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-01-30 19:56:50,385 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


cobrakbase 0.4.0


2026-01-30 19:56:50,616 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: argo


Loaded reference flux from lactate condition:
  Total reactions with flux data: 990
  Non-zero fluxes: 432
  Growth rate: 3.4164642792320175

Sample fluxes (first 10):
  rxn12357_c0: -0.040891
  rxn00947_c0: -0.074948
  rxn00100_c0: 0.014338
  rxn00305_c0: -1.391263
  rxn04783_c0: 0.000000
  rxn15467_c0: 3.297671
  rxn00199_c0: 0.000000
  rxn01353_c0: 0.071539
  rxn03887_c0: 0.000000
  rxn12353_c0: 0.019423

Saved reference flux to datacache/


## Run Fold Change Flux Analysis for All Conditions

This step:
1. For each target condition (strain), computes fold changes relative to ADP1
2. Uses fit_flux_to_proteomics_fold_change_data to fit fluxes
3. Collects results including:
   - Fitted flux distributions
   - Reactions with significant flux changes
   - Proteins whose fold changes could/could not be implemented
   - Reaction matching statistics

In [2]:
%run util.py
import pandas as pd
import cobra.io

# Load saved data
strain_info = util.load("fold_change_strains")
averaged_expression_data = util.load("fold_change_averaged_expression")
reference_flux = util.load("fold_change_reference_flux")

# Load model
model = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")
model.model.reactions.get_by_id("rxn01332_c0").lower_bound = 0
model.model.reactions.get_by_id("rxn01332_c0").upper_bound = 0
model.model.reactions.get_by_id("DgoA").lower_bound = -1000
model.model.reactions.get_by_id("DgoA").upper_bound = 1000

# Rebuild MSExpression from saved data
# Convert dict to DataFrame and reset index to make gene IDs a column
expression_df = pd.DataFrame.from_dict(averaged_expression_data)
expression_df = expression_df.reset_index()
expression_df = expression_df.rename(columns={'index': 'gene_id'})

print(f"Expression DataFrame shape: {expression_df.shape}")
print(f"Columns: {list(expression_df.columns)}")

# Create MSExpression from DataFrame
expression = MSExpression.from_dataframe(
    genome_or_model=util.get_msgenome_from_dict(util.load("ADP1Genome")["data"]),
    df=expression_df,
    id_column='gene_id',
    type="Log2",
    create_missing_features=True
)

print(f"MSExpression created with {len(expression.features)} features and {len(expression.conditions)} conditions")
print(f"Conditions: {[c.id for c in expression.conditions]}")

reference_condition = strain_info["reference_condition"]
target_conditions = strain_info["target_conditions"]

print(f"\nReference condition: {reference_condition}")
print(f"Analyzing {len(target_conditions)} target conditions")
print("=" * 80)

# Store all results
fold_change_results = {}

for target_condition in target_conditions:
    print(f"\nProcessing: {target_condition}")
    print("-" * 60)
    
    try:
        # Create a fresh model copy for each condition
        model_copy = MSModelUtil.from_cobrapy(cobra.io.json.to_json(model.model))
        
        # Run fold change flux fitting
        result = expression.fit_flux_to_proteomics_fold_change_data(
            model=model_copy,
            reference_condition=reference_condition,
            target_condition=target_condition,
            reference_flux=reference_flux,
            zero_flux=0.001,
            least_squares=True,
            fold_change_thresholds=[1.0, 2.0, 3.0],
            quadratic_formulation=True  # 2x, 4x, 8x fold changes
        )
        
        # Extract key statistics
        stats = result['statistics']
        print(f"  Reactions with fold change data: {stats['total_reactions_with_fold_change']}")
        print(f"  Genes with fold change data: {stats['total_genes_with_fold_change']}")
        print(f"  Reactions matched: {stats['reactions_matched']}")
        print(f"  Reactions not matched: {stats['reactions_not_matched']}")
        print(f"  Proteins implemented: {stats['proteins_implemented']}")
        print(f"  Proteins not implemented: {stats['proteins_not_implemented']}")
        
        # Store result (convert solution to serializable format)
        fold_change_results[target_condition] = {
            'fluxes': result['solution'].fluxes.to_dict() if result['solution'] else {},
            'objective_value': result['solution'].objective_value if result['solution'] else None,
            'status': result['solution'].status if result['solution'] else 'error',
            'target_flux': result['target_flux'],
            'fold_changes': {k: v for k, v in result['fold_changes'].items()},
            'gene_fold_changes': result['gene_fold_changes'],
            'flux_changes': {
                'increased_1std': [r['rxn_id'] for r in result['flux_changes']['increased_1std']],
                'increased_2std': [r['rxn_id'] for r in result['flux_changes']['increased_2std']],
                'increased_3std': [r['rxn_id'] for r in result['flux_changes']['increased_3std']],
                'decreased_1std': [r['rxn_id'] for r in result['flux_changes']['decreased_1std']],
                'decreased_2std': [r['rxn_id'] for r in result['flux_changes']['decreased_2std']],
                'decreased_3std': [r['rxn_id'] for r in result['flux_changes']['decreased_3std']]
            },
            'protein_changes': result['protein_changes'],
            'reaction_matching': result['reaction_matching'],
            'statistics': result['statistics']
        }
        
        print(f"  Status: SUCCESS")
        
    except Exception as e:
        print(f"  ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        fold_change_results[target_condition] = {
            'status': 'error',
            'error': str(e)
        }

# Save all results
util.save("ADP1FoldChangeAnalysis", fold_change_results)

print("\n" + "=" * 80)
print(f"Completed analysis for {len(fold_change_results)} conditions")
print("Results saved to datacache/ADP1FoldChangeAnalysis.json")

2026-01-30 20:04:07,521 - __main__.NotebookUtil - INFO - Loaded configuration from: /home/chenry/.kbutillib/config.yaml
2026-01-30 20:04:07,522 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /home/chenry/.kbase/token
2026-01-30 20:04:07,524 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /home/chenry/Dropbox/Projects/ModelSEEDDatabase


/home/chenry/Dropbox/Projects/KBUtilLib/src


2026-01-30 20:04:08,167 - __main__.NotebookUtil - INFO - BLAST tools are available
2026-01-30 20:04:08,168 - __main__.NotebookUtil - INFO - Notebook name: ADP1FoldChangeAnalysis
2026-01-30 20:04:08,169 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-01-30 20:04:08,169 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/
2026-01-30 20:04:08,190 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: argo
2026-01-30 20:04:08,551 - __main__.NotebookUtil - INFO - File not found in /home/chenry/Dropbox/Projects/ADP1Notebooks/notebooks/datacache/ADP1FoldChangeAnalysis, loading from base datacache


Expression DataFrame shape: (2383, 8)
Columns: ['gene_id', 'Pyruvate_ACN2586_DgoA_2025', 'Pyruvate_ACN2821_DgoA_2025', 'Pyruvate_ACN3425_DgoA_2025', 'Pyruvate_ACN3427_DgoA_2025', 'Pyruvate_ACN3429_DgoA_2025', 'Pyruvate_ACN3430_DgoA_2025', 'Pyruvate_ADP1_DgoA_2025']
MSExpression created with 2383 features and 7 conditions
Conditions: ['Pyruvate_ACN2586_DgoA_2025', 'Pyruvate_ACN2821_DgoA_2025', 'Pyruvate_ACN3425_DgoA_2025', 'Pyruvate_ACN3427_DgoA_2025', 'Pyruvate_ACN3429_DgoA_2025', 'Pyruvate_ACN3430_DgoA_2025', 'Pyruvate_ADP1_DgoA_2025']

Reference condition: Pyruvate_ADP1_DgoA_2025
Analyzing 6 target conditions

Processing: Pyruvate_ACN2586_DgoA_2025
------------------------------------------------------------
  Reactions with fold change data: 721
  Genes with fold change data: 2383
  Reactions matched: 411
  Reactions not matched: 310
  Proteins implemented: 33
  Proteins not implemented: 427
  Status: SUCCESS

Processing: Pyruvate_ACN2821_DgoA_2025
----------------------------------

## Summary Statistics

This step:
1. Loads the fold change analysis results
2. Generates summary tables comparing all conditions
3. Identifies common patterns across strains

In [3]:
%run util.py
import pandas as pd

# Load results
results = util.load("ADP1FoldChangeAnalysis")

print("FOLD CHANGE ANALYSIS SUMMARY")
print("=" * 100)

# Build summary table
summary_data = []
for condition, data in results.items():
    if data.get('status') == 'error':
        summary_data.append({
            'Condition': condition,
            'Status': 'ERROR',
            'Rxns w/ FC': 'N/A',
            'Genes w/ FC': 'N/A',
            'Matched': 'N/A',
            'Not Matched': 'N/A',
            'Proteins Impl': 'N/A',
            'Proteins Not Impl': 'N/A',
            'Inc 2x': 'N/A',
            'Dec 2x': 'N/A'
        })
    else:
        stats = data.get('statistics', {})
        fc = data.get('flux_changes', {})
        summary_data.append({
            'Condition': condition.replace('Pyruvate_', '').replace('_DgoA_2025', ''),
            'Status': data.get('status', 'unknown'),
            'Rxns w/ FC': stats.get('total_reactions_with_fold_change', 0),
            'Genes w/ FC': stats.get('total_genes_with_fold_change', 0),
            'Matched': stats.get('reactions_matched', 0),
            'Not Matched': stats.get('reactions_not_matched', 0),
            'Proteins Impl': stats.get('proteins_implemented', 0),
            'Proteins Not Impl': stats.get('proteins_not_implemented', 0),
            'Inc 2x': len(fc.get('increased_1std', [])),
            'Dec 2x': len(fc.get('decreased_1std', []))
        })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Save summary to Excel
summary_df.to_excel("nboutput/fold_change_summary.xlsx", index=False)
print("\nSummary saved to nboutput/fold_change_summary.xlsx")

2026-01-30 20:06:47,162 - __main__.NotebookUtil - INFO - Loaded configuration from: /home/chenry/.kbutillib/config.yaml
2026-01-30 20:06:47,163 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /home/chenry/.kbase/token
2026-01-30 20:06:47,164 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /home/chenry/Dropbox/Projects/ModelSEEDDatabase


/home/chenry/Dropbox/Projects/KBUtilLib/src


2026-01-30 20:06:47,800 - __main__.NotebookUtil - INFO - BLAST tools are available
2026-01-30 20:06:47,802 - __main__.NotebookUtil - INFO - Notebook name: ADP1FoldChangeAnalysis
2026-01-30 20:06:47,803 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-01-30 20:06:47,803 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/
2026-01-30 20:06:47,829 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: argo


FOLD CHANGE ANALYSIS SUMMARY
Condition  Status  Rxns w/ FC  Genes w/ FC  Matched  Not Matched  Proteins Impl  Proteins Not Impl  Inc 2x  Dec 2x
  ACN2586 optimal         721         2383      411          310             33                427     315      34
  ACN2821 optimal         721         2383      448          273             12                298     336      30
  ACN3425 optimal         721         2383      457          264             14                290     343      35
  ACN3427 optimal         721         2383      441          280             22                365     339      38
  ACN3429 optimal         721         2383      469          252              9                272     331      15
  ACN3430 optimal         721         2383      448          273             13                309     357      15

Summary saved to nboutput/fold_change_summary.xlsx


## Protein Implementation Analysis

This step:
1. Identifies proteins with significant fold changes that could not be implemented
2. Analyzes why certain fold changes couldn't be matched in the flux solution
3. Identifies common bottlenecks across strains

In [4]:
%run util.py

# Load results
results = util.load("ADP1FoldChangeAnalysis")

print("PROTEINS WITH SIGNIFICANT FOLD CHANGES NOT IMPLEMENTED")
print("=" * 100)

# Collect proteins not implemented across all conditions
all_not_implemented = {}

for condition, data in results.items():
    if data.get('status') == 'error':
        continue
    
    short_name = condition.replace('Pyruvate_', '').replace('_DgoA_2025', '')
    not_impl = data.get('protein_changes', {}).get('significant_not_implemented', {})
    
    print(f"\n{short_name}: {len(not_impl)} proteins not implemented")
    
    # Show top 10 by fold change magnitude
    sorted_proteins = sorted(not_impl.items(), key=lambda x: abs(x[1].get('log2_fc', 0)), reverse=True)
    
    for gene_id, info in sorted_proteins[:10]:
        log2_fc = info.get('log2_fc', 0)
        reason = info.get('reason', 'unknown')
        direction = 'UP' if log2_fc > 0 else 'DOWN'
        print(f"  {gene_id}: {direction} {abs(log2_fc):.2f} log2FC - {reason}")
        
        # Track across conditions
        if gene_id not in all_not_implemented:
            all_not_implemented[gene_id] = []
        all_not_implemented[gene_id].append({
            'condition': short_name,
            'log2_fc': log2_fc,
            'reason': reason
        })

# Find proteins consistently not implemented
print("\n" + "=" * 100)
print("PROTEINS NOT IMPLEMENTED IN MULTIPLE CONDITIONS")
print("=" * 100)

for gene_id, occurrences in sorted(all_not_implemented.items(), key=lambda x: len(x[1]), reverse=True):
    if len(occurrences) >= 3:
        conditions = [o['condition'] for o in occurrences]
        avg_fc = sum(o['log2_fc'] for o in occurrences) / len(occurrences)
        print(f"{gene_id}: Not implemented in {len(occurrences)} conditions, avg log2FC={avg_fc:.2f}")
        print(f"  Conditions: {', '.join(conditions)}")

2026-01-30 20:09:30,590 - __main__.NotebookUtil - INFO - Loaded configuration from: /home/chenry/.kbutillib/config.yaml
2026-01-30 20:09:30,591 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /home/chenry/.kbase/token
2026-01-30 20:09:30,592 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /home/chenry/Dropbox/Projects/ModelSEEDDatabase


/home/chenry/Dropbox/Projects/KBUtilLib/src


2026-01-30 20:09:31,239 - __main__.NotebookUtil - INFO - BLAST tools are available
2026-01-30 20:09:31,241 - __main__.NotebookUtil - INFO - Notebook name: ADP1FoldChangeAnalysis
2026-01-30 20:09:31,242 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-01-30 20:09:31,242 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/
2026-01-30 20:09:31,263 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: argo


PROTEINS WITH SIGNIFICANT FOLD CHANGES NOT IMPLEMENTED

ACN2586: 427 proteins not implemented
  DgoA: UP 9.15 log2FC - flux_could_not_match
  DgoA_Ec: UP 8.09 log2FC - flux_could_not_match
  ACIAD_RS13800: DOWN 6.09 log2FC - no_reactions_in_model
  ACIAD_RS12895: UP 5.01 log2FC - no_reactions_in_model
  ACIAD_RS05915: DOWN 4.74 log2FC - flux_could_not_match
  ACIAD_RS10945: UP 4.29 log2FC - no_reactions_in_model
  ACIAD_RS00450: DOWN 4.27 log2FC - flux_could_not_match
  ACIAD_RS06160: UP 3.86 log2FC - flux_could_not_match
  ACIAD_RS15770: DOWN 3.68 log2FC - no_reactions_in_model
  ACIAD_RS03785: DOWN 3.61 log2FC - no_reactions_in_model

ACN2821: 298 proteins not implemented
  DgoA: UP 8.65 log2FC - flux_could_not_match
  DgoA_Ec: UP 7.75 log2FC - flux_could_not_match
  ACIAD_RS15555: UP 5.76 log2FC - no_reactions_in_model
  ACIAD_RS04325: UP 5.10 log2FC - no_reactions_in_model
  ACIAD_RS05600: DOWN 4.75 log2FC - no_reactions_in_model
  ACIAD_RS00450: DOWN 4.64 log2FC - flux_could_not_m

## Visualize Reference Flux on Escher Map

This step:
1. Loads the reference flux from lactate condition
2. Visualizes it on the full.json Escher map
3. Generates an interactive HTML file for exploring the flux distribution

In [ ]:
%run util.py

# Load reference flux
reference_flux = util.load("fold_change_reference_flux")

# Load the model for metadata
model = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")

print(f"Reference flux has {len(reference_flux)} reactions")
print(f"Non-zero fluxes: {sum(1 for v in reference_flux.values() if abs(v) > 1e-9)}")

# Create Escher map visualization for reference flux
output_file = util.create_map_html2(
    model=model.model,
    flux=reference_flux,
    map="full",
    output_path=util.self.output_dir+"/escher_reference_flux.html",
)

print(f"\nReference flux map saved to: {output_file}")

# Display link to open the map
from IPython.display import HTML, display
display(HTML(f'<a href="{output_file}" target="_blank">Open Reference Flux Map</a>'))

## Visualize Fitted Flux for Selected Condition

This step:
1. Allows selection of a target condition to visualize
2. Loads the fitted flux from fold change analysis
3. Displays on Escher map to compare with reference

In [ ]:
%run util.py

# Load fold change results
results = util.load("ADP1FoldChangeAnalysis")

# Select condition to visualize (change this to view different conditions)
selected_condition = "Pyruvate_ACN2586_DgoA_2025"  # Options: ACN2586, ACN2821, ACN3425, ACN3427, ACN3429, ACN3430

# Get short name for display
short_name = selected_condition.replace('Pyruvate_', '').replace('_DgoA_2025', '')

print(f"Selected condition: {short_name}")
print(f"Available conditions: {list(results.keys())}")

# Get fitted flux for selected condition
condition_data = results[selected_condition]
if condition_data.get('status') == 'error':
    print(f"ERROR: {condition_data.get('error', 'Unknown error')}")
else:
    fitted_flux = condition_data['fluxes']
    
    print(f"\nFitted flux has {len(fitted_flux)} reactions")
    print(f"Non-zero fluxes: {sum(1 for v in fitted_flux.values() if abs(v) > 1e-9)}")
    
    # Load model for metadata
    model = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")
    
    # Create Escher map visualization for fitted flux
    output_file = util.create_map_html2(
        model=model.model,
        flux=fitted_flux,
        map="full",
        output_path=util.self.output_dir+f"/escher_fitted_flux_{short_name}.html",
    )
    
    print(f"\nFitted flux map saved to: {output_file}")
    
    # Display link
    from IPython.display import HTML, display
    display(HTML(f'<a href="{output_file}" target="_blank">Open Fitted Flux Map ({short_name})</a>'))

## Visualize Flux Differential (Fitted - Reference)

This step:
1. Computes the flux differential between fitted and reference fluxes
2. Visualizes on Escher map using directional color scheme
3. Positive values (green) = increased flux, Negative values (red) = decreased flux

In [6]:
%run util.py

# Load data
results = util.load("ADP1FoldChangeAnalysis")
reference_flux = util.load("fold_change_reference_flux")

# Select condition to compare (change this to view different conditions)
selected_condition = "Pyruvate_ACN2586_DgoA_2025"  # Options: ACN2586, ACN2821, ACN3425, ACN3427, ACN3429, ACN3430

# Get short name for display
short_name = selected_condition.replace('Pyruvate_', '').replace('_DgoA_2025', '')

print(f"Computing flux differential for: {short_name}")

# Get fitted flux for selected condition
condition_data = results[selected_condition]
if condition_data.get('status') == 'error':
    print(f"ERROR: {condition_data.get('error', 'Unknown error')}")
else:
    fitted_flux = condition_data['fluxes']
    
    # Compute flux differential (fitted - reference)
    flux_differential = {}
    all_rxns = set(fitted_flux.keys()) | set(reference_flux.keys())
    
    for rxn_id in all_rxns:
        fitted_val = fitted_flux.get(rxn_id, 0)
        ref_val = reference_flux.get(rxn_id, 0)
        diff = fitted_val - ref_val
        flux_differential[rxn_id] = diff
    
    # Statistics on differential
    positive_changes = sum(1 for v in flux_differential.values() if v > 0.01)
    negative_changes = sum(1 for v in flux_differential.values() if v < -0.01)
    unchanged = len(flux_differential) - positive_changes - negative_changes
    
    print(f"\nFlux differential statistics:")
    print(f"  Increased (>0.01): {positive_changes} reactions")
    print(f"  Decreased (<-0.01): {negative_changes} reactions")
    print(f"  Unchanged: {unchanged} reactions")
    
    # Show top 10 largest changes
    sorted_diffs = sorted(flux_differential.items(), key=lambda x: abs(x[1]), reverse=True)
    print(f"\nTop 10 largest flux changes:")
    for rxn_id, diff in sorted_diffs[:10]:
        direction = "+" if diff > 0 else ""
        print(f"  {rxn_id}: {direction}{diff:.4f}")
    
    # Load model for metadata
    model = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")
    
    # Create Escher map visualization for flux differential
    # Using directional color scheme: red = decreased, green = increased
    output_file = util.create_map_html2(
        model=model.model,
        flux=flux_differential,
        map="full",
        output_path=util.output_dir+f"/escher_fitted_flux_{short_name}.html",
    )
    
    print(f"\nFlux differential map saved to: {output_file}")
    
    # Display link
    from IPython.display import HTML, display
    display(HTML(f'<a href="{output_file}" target="_blank">Open Flux Differential Map ({short_name})</a>'))

2026-01-31 14:12:15,445 - __main__.NotebookUtil - INFO - Loaded configuration from: /home/chenry/.kbutillib/config.yaml
2026-01-31 14:12:15,446 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /home/chenry/.kbase/token
2026-01-31 14:12:15,447 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /home/chenry/Dropbox/Projects/ModelSEEDDatabase


/home/chenry/Dropbox/Projects/KBUtilLib/src


2026-01-31 14:12:16,090 - __main__.NotebookUtil - INFO - BLAST tools are available
2026-01-31 14:12:16,092 - __main__.NotebookUtil - INFO - Notebook name: ADP1FoldChangeAnalysis
2026-01-31 14:12:16,093 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-01-31 14:12:16,093 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/
2026-01-31 14:12:16,113 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: argo


Computing flux differential for: ACN2586

Flux differential statistics:
  Increased (>0.01): 278 reactions
  Decreased (<-0.01): 421 reactions
  Unchanged: 291 reactions

Top 10 largest flux changes:
  rxn00062_c0: +288.0542
  rxn00086_c0: -283.6905
  rxn00205_c0: +283.6905
  rxn00262_c0: +182.9339
  UNBAL_DASH_PROTON: +141.4384
  rxn00161_c0: +140.7093
  EXF_DASH_AMMONIUM_LBRACKET_Extraorganism_RBRACKET_: +114.1591
  rxn05466_c0: -114.1591
  rxn00154_c0: +103.0811
  EXF_DASH_OXYGEN_DASH_MOLECULE_LBRACKET_Extraorganism_RBRACKET_: -100.0000


ImportError: escher package is required for map visualization. Install with: pip install escher

## Generate All Condition Maps

This step:
1. Generates Escher maps for all conditions in a single batch
2. Creates fitted flux maps and flux differential maps for each strain
3. Provides links to all generated visualizations

In [7]:
%run util.py

# Load all data
results = util.load("ADP1FoldChangeAnalysis")
reference_flux = util.load("fold_change_reference_flux")

# Load model once for all visualizations
model = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")

print("Generating Escher maps for all conditions...")
print("=" * 80)

generated_files = {
    'reference': None,
    'fitted': {},
    'differential': {}
}

# Generate reference flux map first
print("\n[Reference Flux]")
ref_output = util.create_map_html2(
    model=model.model,
    flux=reference_flux,
    map="full",
    output_path=util.output_dir+"/escher_reference_flux.html",
)
generated_files['reference'] = ref_output
print(f"  Saved: {ref_output}")

# Generate maps for each condition
for condition, data in results.items():
    short_name = condition.replace('Pyruvate_', '').replace('_DgoA_2025', '')
    print(f"\n[{short_name}]")
    
    if data.get('status') == 'error':
        print(f"  SKIPPED - Error in analysis")
        continue
    
    fitted_flux = data['fluxes']
    
    # Generate fitted flux map
    fitted_output = util.create_map_html2(
        model=model.model,
        flux=fitted_flux,
        map="full",
        output_path=util.output_dir+f"/escher_fitted_flux_{short_name}.html"
    )

    #fitted_output = util.create_map_html(
    #    model=model.model,
    #    flux_solution=fitted_flux,
    #    map_json="data/full.json",
    #    output_file=f"nboutput/escher_fitted_flux_{short_name}.html",
    #    title=f"Fitted Flux - {short_name}",
    #    enhanced_arrows=True,
    #    arrow_color_scheme='magnitude'
    #)
    #generated_files['fitted'][short_name] = fitted_output
    #print(f"  Fitted flux: {fitted_output}")
    
    # Compute and generate differential map
    flux_differential = {}
    all_rxns = set(fitted_flux.keys()) | set(reference_flux.keys())
    for rxn_id in all_rxns:
        diff = fitted_flux.get(rxn_id, 0) - reference_flux.get(rxn_id, 0)
        flux_differential[rxn_id] = diff
    
    #diff_output = util.create_map_html(
    #    model=model.model,
    #    flux_solution=flux_differential,
    #    map_json="data/full.json",
    #    output_file=f"nboutput/escher_flux_diff_{short_name}.html",
    #    title=f"Flux Differential ({short_name} - Reference)",
    #    enhanced_arrows=True,
    #    arrow_color_scheme='directional'
    #)
    #generated_files['differential'][short_name] = diff_output
    #print(f"  Differential: {diff_output}")

# Save file list for reference
util.save("fold_change_escher_files", generated_files)

print("\n" + "=" * 80)
print("All maps generated successfully!")
print(f"Total files: 1 reference + {len(generated_files['fitted'])} fitted + {len(generated_files['differential'])} differential")

# Display links to all files
from IPython.display import HTML, display

html_links = "<h3>Generated Escher Maps</h3>"
html_links += "<h4>Reference Flux:</h4>"
html_links += f'<p><a href="{generated_files["reference"]}" target="_blank">Reference Flux (Lactate)</a></p>'

html_links += "<h4>Fitted Flux Maps:</h4><ul>"
for name, path in generated_files['fitted'].items():
    html_links += f'<li><a href="{path}" target="_blank">{name}</a></li>'
html_links += "</ul>"

html_links += "<h4>Flux Differential Maps (vs Reference):</h4><ul>"
for name, path in generated_files['differential'].items():
    html_links += f'<li><a href="{path}" target="_blank">{name}</a></li>'
html_links += "</ul>"

display(HTML(html_links))

2026-01-31 14:13:37,769 - __main__.NotebookUtil - INFO - Loaded configuration from: /home/chenry/.kbutillib/config.yaml
2026-01-31 14:13:37,771 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /home/chenry/.kbase/token
2026-01-31 14:13:37,772 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /home/chenry/Dropbox/Projects/ModelSEEDDatabase


/home/chenry/Dropbox/Projects/KBUtilLib/src


2026-01-31 14:13:38,420 - __main__.NotebookUtil - INFO - BLAST tools are available
2026-01-31 14:13:38,421 - __main__.NotebookUtil - INFO - Notebook name: ADP1FoldChangeAnalysis
2026-01-31 14:13:38,423 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-01-31 14:13:38,423 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/
2026-01-31 14:13:38,443 - __main__.NotebookUtil - INFO - AICurationUtils initialized with backend: argo


KeyboardInterrupt: 